# Native periodic Ising optimization

Minimize the source energy E=-10 sum(s_i s_(i+1))+20 sum(s_i), with six spins, periodic boundary conditions and s_i=2*b_i-1. Preserve five QAOA layers, lower-tail quantile 0.7, 100 optimization evaluations, convergence plot and optimized/uniform weighted histograms. Write quantum intent with qfunc; let synth own gates and physical wires.

The frozen exported Qmod cost is -E on every one of 64 configurations. Keep that counterpart unchanged and negate its gamma parameters to express source-energy phases. This is an explicit objective-sign correction, not a claim that source and exported classical costs agree. The complete native Circuit reference independently constructs E from periodic edge terms.


In [ ]:
from examples.synthesis.applications.physical_systems.ising_model.overview.workflow import (
    energy,
    optimize,
)

print(
    "Source ground energy (validation only):", min(energy(word) for word in range(64))
)
optimization = optimize()
print("Initial cost:", optimization.initial_cost)
print("Best visited cost:", optimization.best_cost)
print("Termination:", optimization.termination)
assert len(optimization.trace) == 100
assert optimization.best_cost < optimization.initial_cost


## One native program for coherent state preparation, costs and mixing

The optimizer uses exact native probability marginals for a deterministic local lower-tail objective. Each evaluation synthesizes and executes a five-layer native program. An explicit bounded coordinate search replaces Classiq's unspecified private optimizer: deterministic starting angles, coordinate steps and 100 evaluations are visible in workflow.py. No known optimum is given to the search. A budget stop is not convergence certification.

Below, synthesize the same best visited ansatz directly. A quantum array readout returns a named integer configuration; physical wire numbers and bitstrings are not part of the user flow. The source energy's symbolic phase is applied by reusing the existing checked cost_layer with a negative gamma.


In [ ]:
from flagquantum import synth
from flagquantum.synth import (
    Output,
    QArray,
    QBit,
    allocate,
    hadamard_transform,
    RX,
    qfunc,
)
from examples.synthesis.applications.physical_systems.ising_model.main import cost_layer
from examples.synthesis.applications.physical_systems.ising_model.overview.workflow import (
    source_program,
)


@qfunc
def ising_program(params: list[float], v: Output[QArray[QBit]]) -> None:
    allocate(6, v)
    hadamard_transform(v)
    for layer in range(5):
        cost_layer(-params[layer], v)
        for spin in range(6):
            RX(params[5 + layer], v[spin])


program = synth.synthesize(
    ising_program,
    classical_inputs={"params": optimization.parameters},
    max_qubits=6,
    max_workspace=0,
)
uniform_program = source_program((0.0,) * 10)
print(program.report["resources"])


## Actual finite samples and best observed spin configurations

Execute optimized and uniform programs, then draw 2048 local samples from each exact native distribution with predetermined seeds 123 and 456. Preserve zero-count configurations. Display the best five observed configurations ranked by source energy, with spins, counts and finite probabilities. The exact optimization objective and finite output samples are separate. The uniformly prepared quantum state is executed rather than replacing it with an assumed histogram. Neither a statevector simulation nor a classical local optimizer demonstrates hardware advantage.


In [ ]:
from examples.synthesis.applications.physical_systems.ising_model.overview.main import (
    read_workflow,
)

workflow = read_workflow(optimization, program, uniform_program)
assert sum(row.count for row in workflow.optimized_samples) == 2048
assert sum(row.count for row in workflow.uniform_samples) == 2048
observed = [row for row in workflow.optimized_samples if row.count]
best_observed = min(observed, key=lambda row: row.energy)
print("Best observed spins:", best_observed.spins)
print("Best observed energy:", best_observed.energy)
assert best_observed.energy == min(energy(word) for word in range(64))


## Convergence and weighted energy histograms

Keep both source plots. The convergence trace contains all evaluated exact quantile costs, including rejected candidates; it is not forced monotone. Energy histograms use the actual finite sample probabilities and30 common bins so optimized and uniform rows are directly comparable. A tail cost equal to the minimum does not imply a pure ground state: report the observed ground probability separately. The known minimum is used only for validation.


In [ ]:
from examples.synthesis.applications.physical_systems.ising_model.overview.plots import (
    plot_results,
)

convergence, histogram = plot_results(
    workflow.optimization, workflow.optimized_samples, workflow.uniform_samples
)
minimum = min(energy(word) for word in range(64))
print(
    "Observed ground-state probability:",
    sum(row.probability for row in workflow.optimized_samples if row.energy == minimum),
)
